# Regresión de soporte vectorial

Una lectura de  cómo el método de soporte vectorial también se puede usar para la regresión.

https://www.saedsayad.com/support_vector_machine_reg.htm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Una función no lineal simple
El objetivo es crear algunos datos sintéticos que no sean muy adecuados para los modelos de regresión lineal. Mostraremos cómo un regresor de Vector de soporte mejora el rendimiento predictivo.

In [ ]:
def nonlinear(array):
    return (10*array[:,0]-np.exp(0.01*array[:,1]+np.log(1+array[:,2]**2)))/(array[:,3]**2+5)

### Generar características y datos de destino para la regresión

In [ ]:
n_samples = 200
n_features = 4

In [ ]:
x = 5*np.random.rand(n_samples,n_features)
x.shape

In [ ]:
y = nonlinear(x)+np.random.randn(n_samples)
y.shape


In [ ]:
y=y.reshape(n_samples,1)
y.shape

In [ ]:
df = pd.DataFrame(data=np.hstack((x,y)),columns=['X1','X2','X3','X4','y'])

In [ ]:
df.head()

### Visualizando los datos

In [ ]:
fig,ax = plt.subplots(2,2,figsize=(10,8))
ax = ax.ravel()
for i in range(4):
    ax[i].scatter(df[df.columns[i]],df['y'],edgecolor='k',color='red',alpha=0.75)
    ax[i].set_title(f"{df.columns[i]} vs. y",fontsize=14)
    ax[i].grid(True)
plt.show()

### División de entrenamiento/prueba

In [ ]:
X = df[['X1','X2','X3','X4']]
y = df['y']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

### Regreso de soporte vectorial con kernel lineal

El tutorial de SVR de scikit-learn: https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html

In [ ]:
from sklearn.svm import SVR
svr_linear = SVR(kernel='linear',gamma='scale', C=1.0, epsilon=0.2)
svr_linear.fit(X_train, y_train)

### Test score

In [ ]:
svr_linear.score(X_test,y_test)

### Regresión lineal como línea de base

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
linear = LinearRegression()

In [ ]:
linear.fit(X_train,y_train)

In [ ]:
linear.score(X_test,y_test)

### Regresor de soporte vectorial con kernel Gaussiano (radial basis function)

In [ ]:
from sklearn.kernel_approximation import RBFSampler

X1 = [[0, 0], [1, 1], [1, 0], [0, 1]]
print(len(X1))
y = [0, 0, 1, 1]
rbf_feature = RBFSampler(gamma=1, random_state=1)
X_features1 = rbf_feature.fit_transform(X1)
print(X_features1)


In [ ]:
from sklearn.kernel_approximation import RBFSampler
import seaborn as sns
import pandas as pd
rbf_feature = RBFSampler(gamma=1, random_state=1)
#X_train.shape
x_features = rbf_feature.fit_transform(X_train)
#X_features
x_features.shape
#plt.scatter(X_features,y_train,c='green',edgecolors='k')


In [ ]:
svr_rbf = SVR(kernel='rbf',gamma='scale', C=1.0, epsilon=0.1)
svr_rbf.fit(X_train, y_train)

In [ ]:
svr_rbf.score(X_test,y_test)

Entonces, claramente, el kernel RBF mostró una mejor precisión en el conjunto de prueba

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
print("RMSE para linear SVR:",np.sqrt(mean_squared_error(y_test,svr_linear.predict(X_test))))
print("RMSE para RBF kernelized SVR:",np.sqrt(mean_squared_error(y_test,svr_rbf.predict(X_test))))

### Podemos hacer una búsqueda en cuadrícula de hiperparámetros (con validación cruzada de 5 veces) para ver si se mejora la puntuación de la prueba/validación

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
params = {'C':[0.01,0.05,0.1,0.5,1,2,5],'epsilon':[0.1,0.2,0.5,1]}

In [ ]:
grid = GridSearchCV(svr_rbf,param_grid=params,cv=5,scoring='r2',verbose=1,return_train_score=True)

In [ ]:
grid.fit(X_train,y_train)

### Verifique cuál fue el mejor estimador según la búsqueda de cuadrícula

In [ ]:
grid.best_estimator_

### Ajuste ese estimador a los datos y vea

In [ ]:
svr_best=SVR(kernel='rbf',gamma='scale', C=2, epsilon=0.2)
svr_best.fit(X_train, y_train)

In [ ]:
svr_best.score(X_test,y_test)

In [ ]:
print("RMSE para RBF kernelized SVR:",np.sqrt(mean_squared_error(y_test,svr_best.predict(X_test))))